# Causal Inference Limitations: Case Studies

When researchers claim "media caused X," how should we evaluate that claim? This notebook explores data from Bailard et al. (2024, APSR) to see what observational evidence can and cannot tell us.

1. Load and explore the Proud Boys Telegram + offline events data
2. Reproduce the key finding: grievance messages predict offline violence
3. Ask: is this causal? What would we need to make that claim?

Data source: [Harvard Dataverse](https://doi.org/10.7910/DVN/OAOJQZ) (CC0 license)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. The Data: Proud Boys Telegram Messages + Offline Events

Bailard et al. (2024) scraped 514,000 messages from Proud Boys Telegram channels (Jan 2020 to July 2022). They trained ML classifiers to categorize each message:

- **Diagnostic**: identifying grievances, naming enemies
- **Prognostic**: proposing solutions, calls to action
- **Motivational**: appeals to group pride, solidarity
- **Othering**: dehumanizing language about outgroups

They then matched weekly message counts to the US Crisis Monitor's data on offline events (protests and violent incidents).

In [ ]:
# Load weekly time-series data (134 weeks)
DATA_URL = 'https://raw.githubusercontent.com/Persuasion-at-Scale/causal-limitations/main/data/bailard-ts-data.tab'
LOCAL_PATH = '../data/bailard-ts-data.tab'

import os
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH, sep='\t')
else:
    df = pd.read_csv(DATA_URL, sep='\t')

df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

print(f"Weeks: {len(df)} (from {df.start_date.min().date()} to {df.end_date.max().date()})")
print(f"Total messages classified: {df.number_texts_all.sum():,.0f}")
print(f"Total violent events: {df.violent_event.sum():.0f}")
print(f"Total non-violent protests: {df.non_violent_protest.sum():.0f}")
print()
print("Message type proportions (weekly averages):")
for col in ['diagnostic', 'prognostic', 'motivational', 'othering']:
    print(f"  {col}: {df[col].mean():.3f}")

## 2. What do the time series look like?

In [ ]:
# Plot message types and offline events over time
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: message composition over time
for col, color, label in [('diagnostic', 'firebrick', 'Grievances'),
                            ('motivational', 'steelblue', 'Group pride'),
                            ('othering', 'darkorange', 'Othering')]:
    axes[0].plot(df['start_date'], df[col], color=color, label=label, alpha=0.7)
axes[0].set_ylabel('Share of messages')
axes[0].set_title('Proud Boys Telegram: message types over time')
axes[0].legend(fontsize=10)

# Bottom: offline events
axes[1].bar(df['start_date'], df['violent_event'], width=5, color='firebrick',
            alpha=0.7, label='Violent events')
axes[1].bar(df['start_date'], df['non_violent_protest'], width=5, color='steelblue',
            alpha=0.4, label='Non-violent protests', bottom=df['violent_event'])
axes[1].set_ylabel('Events per week')
axes[1].set_title('Offline Proud Boys activity (US Crisis Monitor)')
axes[1].legend(fontsize=10)

# Mark Jan 6
jan6 = pd.Timestamp('2021-01-06')
for ax in axes:
    ax.axvline(jan6, color='black', linestyle='--', alpha=0.5)
axes[0].text(jan6, axes[0].get_ylim()[1]*0.9, ' Jan 6', fontsize=10)

plt.tight_layout()
plt.show()

print("The January 6 Capitol attack is visible as the largest spike in violent events.")
print("Notice how grievance messaging (red) and offline violence both peak around the same time.")

## 3. The Key Finding: Grievances Predict Violence

Bailard et al. use Granger causality tests: does *lagged* grievance messaging help predict future violent events, beyond what past violence alone predicts?

Let's reproduce this with a simple regression: does last week's grievance share predict this week's violent events?

In [ ]:
import statsmodels.formula.api as smf

# Create lagged variables
df['grievance_lag1'] = df['diagnostic'].shift(1)
df['grievance_lag2'] = df['diagnostic'].shift(2)
df['violence_lag1'] = df['violent_event'].shift(1)

sample = df.dropna(subset=['grievance_lag1', 'violence_lag1']).copy()

# Model 1: violence predicted by past violence only (autoregressive)
m1 = smf.ols('violent_event ~ violence_lag1', data=sample).fit()

# Model 2: add lagged grievance messages
m2 = smf.ols('violent_event ~ violence_lag1 + grievance_lag1', data=sample).fit()

# Model 3: add second lag
m3 = smf.ols('violent_event ~ violence_lag1 + grievance_lag1 + grievance_lag2', data=sample).fit()

print("Does last week's grievance messaging predict this week's violence?")
print()
print(f"{'Model':<45} {'Coef':>8} {'SE':>8} {'p':>8}")
print("-" * 72)
print(f"{'Violence_lag1 only (autoregressive)':<45} {m1.params['violence_lag1']:>8.3f} {m1.bse['violence_lag1']:>8.3f} {m1.pvalues['violence_lag1']:>8.3f}")
print(f"{'+ Grievance_lag1':<45} {m2.params['grievance_lag1']:>8.3f} {m2.bse['grievance_lag1']:>8.3f} {m2.pvalues['grievance_lag1']:>8.3f}")
print(f"{'+ Grievance_lag1 + lag2':<45} {m3.params['grievance_lag1']:>8.3f} {m3.bse['grievance_lag1']:>8.3f} {m3.pvalues['grievance_lag1']:>8.3f}")
print()
print(f"R-squared: autoregressive={m1.rsquared:.3f}, + grievance={m2.rsquared:.3f}, + 2 lags={m3.rsquared:.3f}")
print()
print("If the grievance coefficient is positive and significant, past online")
print("grievance messaging helps predict future offline violence.")

## 4. But Is This Causal?

The regression above shows *prediction*, not necessarily *causation*. Several problems:

**Omitted variables**: maybe a news event (e.g., a court ruling, a counter-protest) causes both more online anger AND more offline mobilization at the same time.

**Reverse causation**: maybe offline events (arrests, clashes) generate online discussion, not the other way around.

**Shared trends**: both online messaging and offline activity could be driven by the same underlying political cycle (elections, trials, legislation).

What would we *need* for a causal claim? Think back to weeks 8-9:
- An **instrument** (something that shifts online messaging without directly affecting offline violence)
- A **natural experiment** (an exogenous shock to one but not the other)
- An **RCT** (randomly assign exposure to messaging... which is ethically impossible here)

In [ ]:
# Scatter plot: grievance share vs. next week's violence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: lagged grievance vs violence (the "causal" direction)
axes[0].scatter(sample['grievance_lag1'], sample['violent_event'],
                alpha=0.5, color='firebrick', s=30)
axes[0].set_xlabel('Grievance share (previous week)')
axes[0].set_ylabel('Violent events (this week)')
axes[0].set_title('Does past grievance predict future violence?')

# Right: lagged violence vs grievance (reverse causation check)
sample['diagnostic_lead'] = sample['diagnostic']  # current week's grievance
axes[1].scatter(sample['violence_lag1'], sample['diagnostic_lead'],
                alpha=0.5, color='steelblue', s=30)
axes[1].set_xlabel('Violent events (previous week)')
axes[1].set_ylabel('Grievance share (this week)')
axes[1].set_title('Or does past violence predict future grievance?')

plt.tight_layout()
plt.show()

# Check both directions
fwd = smf.ols('violent_event ~ grievance_lag1', data=sample).fit()
rev = smf.ols('diagnostic ~ violence_lag1', data=sample).fit()
print(f"Grievance -> Violence:  coef={fwd.params['grievance_lag1']:.3f}, p={fwd.pvalues['grievance_lag1']:.3f}")
print(f"Violence -> Grievance:  coef={rev.params['violence_lag1']:.4f}, p={rev.pvalues['violence_lag1']:.3f}")
print()
print("If both directions are significant, we can't tell which causes which.")
print("This is the fundamental limitation of observational time-series data.")

## 5. Contrast with IV Papers

In weeks 8-9, we saw papers that *could* make causal claims because they had instruments:

| Paper | Instrument | Why it works |
|-------|-----------|-------------|
| Lelkes et al. (2017) | ROW regulations | Laws about cable-laying are unrelated to partisan attitudes |
| Angrist (1990) | Draft lottery | Literally random assignment |

Bailard et al. have no instrument. There is no exogenous shock that affects Proud Boys online messaging without also affecting their offline behavior. This limits what the paper can claim.

**Prediction is still valuable**: knowing that online grievance messaging predicts future violence could help with early warning systems. You don't need causation for prediction. But you do need it to say "if we shut down the Telegram channels, violence would decrease."

## Key Takeaway

The gap between "X predicts Y" and "X causes Y" is the central challenge of causal inference. The methods we've learned (IV, RDD, DiD) are tools for crossing that gap, but they require specific conditions. When those conditions aren't met, we're left with suggestive evidence, not proof.